In [2]:
"""
05_pca_interpretation.ipynb

Per-city PCA on AlphaEarth embeddings (A00–A63)
Goal: interpret what each PC captures by correlating with observables

Outputs (per city):
  outputs/tables/<city>_pc_scores.csv       — GEOID + PC1-PC7
  outputs/tables/<city>_pc_loadings.csv     — loading of each dim on each PC
  outputs/figures/pca/<city>_pc_variance.png
  outputs/figures/pca/<city>_pc_heatmap.png — top loading dims per PC
  outputs/tables/all_cities_pc_obs_corr.csv — PC vs observables Spearman r
"""

import pandas as pd, numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.stats import spearmanr
import warnings; warnings.filterwarnings("ignore")

ROOT       = Path("..").resolve()
OUT        = ROOT / "outputs"
FIG_PCA    = OUT / "figures" / "pca"
FIG_PCA.mkdir(parents=True, exist_ok=True)
(OUT / "tables").mkdir(exist_ok=True)

df = pd.read_csv(OUT / "modeling_table.csv", dtype={"GEOID": str})
df["GEOID"] = df["GEOID"].str.zfill(11)

EMBED_COLS = [f"A{i:02d}" for i in range(64)]
N_PCS      = 7
CITIES     = sorted(df["city"].unique().tolist())

# Observables to correlate PCs against
OBS_COLS = ["lst_c", "hi_c", "hi_minus_lst_z", "poverty_rate"]

all_corr_rows = []

for city in CITIES:
    sub = df[df["city"] == city].dropna(subset=EMBED_COLS).copy()
    if len(sub) < 20:
        print(f"{city}: skipped"); continue

    print(f"\n{city.upper()}  (n={len(sub)})")

    X = sub[EMBED_COLS].values
    X_scaled = StandardScaler().fit_transform(X)

    pca = PCA(n_components=N_PCS, random_state=42)
    scores = pca.fit_transform(X_scaled)
    evr    = pca.explained_variance_ratio_

    pc_labels = [f"PC{i+1}" for i in range(N_PCS)]
    print(f"  Explained variance: " + "  ".join(f"{pc_labels[i]}={evr[i]:.3f}" for i in range(N_PCS)))
    print(f"  Cumulative (PC1-7): {evr.sum():.3f}")

    # ── PC scores CSV ─────────────────────────────────────────────────────────
    scores_df = pd.DataFrame(scores, columns=pc_labels)
    scores_df.insert(0, "GEOID", sub["GEOID"].values)
    scores_df.to_csv(OUT / "tables" / f"{city}_pc_scores.csv", index=False)

    # ── Loadings CSV ──────────────────────────────────────────────────────────
    loadings_df = pd.DataFrame(
        pca.components_.T, index=EMBED_COLS, columns=pc_labels
    )
    loadings_df.index.name = "embedding_dim"
    loadings_df.to_csv(OUT / "tables" / f"{city}_pc_loadings.csv")

    # ── Explained variance plot ───────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(pc_labels, evr * 100, color="#4C72B0")
    ax.plot(pc_labels, np.cumsum(evr) * 100, "o-", color="#DD8452", label="Cumulative")
    ax.set_ylabel("Explained variance (%)")
    ax.set_title(f"{city.replace('_', ' ').title()} — PCA explained variance")
    ax.legend(); ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIG_PCA / f"{city}_pc_variance.png", dpi=180)
    plt.close()

    # ── Loading heatmap (top 15 dims by max absolute loading) ─────────────────
    max_abs = loadings_df.abs().max(axis=1).sort_values(ascending=False)
    top_dims = max_abs.head(15).index.tolist()
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(
        loadings_df.loc[top_dims],
        cmap="PuOr", center=0, vmin=-0.4, vmax=0.4,
        linewidths=0.3, ax=ax,
        annot=True, fmt=".2f", annot_kws={"size": 7}
    )
    ax.set_title(f"{city.replace('_', ' ').title()} — PC loadings (top 15 dims)")
    plt.tight_layout()
    plt.savefig(FIG_PCA / f"{city}_pc_heatmap.png", dpi=180)
    plt.close()

    # ── Correlate PCs with observables ────────────────────────────────────────
    merged = sub[["GEOID"] + OBS_COLS].copy().reset_index(drop=True)
    merged = pd.concat([merged, pd.DataFrame(scores, columns=pc_labels)], axis=1)

    for pc in pc_labels:
        for obs in OBS_COLS:
            pair = merged[[pc, obs]].dropna()
            if len(pair) < 10: continue
            r, p = spearmanr(pair[pc], pair[obs])
            all_corr_rows.append({
                "city": city, "pc": pc, "observable": obs,
                "spearman_r": round(r, 3), "p_value": round(p, 4),
                "n": len(pair)
            })

    # Print top correlations
    city_corr = pd.DataFrame([x for x in all_corr_rows if x["city"] == city])
    if not city_corr.empty:
        top = city_corr.reindex(city_corr["spearman_r"].abs().sort_values(ascending=False).index).head(5)
        print("  Top PC-observable correlations:")
        for _, row in top.iterrows():
            print(f"    {row['pc']} ↔ {row['observable']:25s} r={row['spearman_r']:+.3f}")

# ── Save cross-city correlation table ────────────────────────────────────────
corr_df = pd.DataFrame(all_corr_rows)
corr_df.to_csv(OUT / "tables" / "all_cities_pc_obs_corr.csv", index=False)

print("\n=== Done ===")
print(f"Variance plots  → outputs/figures/pca/<city>_pc_variance.png")
print(f"Loading heatmaps → outputs/figures/pca/<city>_pc_heatmap.png")
print(f"PC scores CSVs  → outputs/tables/<city>_pc_scores.csv")
print(f"Loadings CSVs   → outputs/tables/<city>_pc_loadings.csv")
print(f"Cross-city corr → outputs/tables/all_cities_pc_obs_corr.csv")



ATLANTA  (n=223)
  Explained variance: PC1=0.465  PC2=0.149  PC3=0.105  PC4=0.059  PC5=0.042  PC6=0.034  PC7=0.025
  Cumulative (PC1-7): 0.879
  Top PC-observable correlations:
    PC1 ↔ lst_c                     r=-0.772
    PC1 ↔ hi_minus_lst_z            r=+0.771
    PC2 ↔ hi_c                      r=-0.643
    PC3 ↔ hi_minus_lst_z            r=+0.370
    PC3 ↔ lst_c                     r=-0.363

BOSTON  (n=191)
  Explained variance: PC1=0.315  PC2=0.256  PC3=0.116  PC4=0.093  PC5=0.057  PC6=0.031  PC7=0.026
  Cumulative (PC1-7): 0.894
  Top PC-observable correlations:
    PC1 ↔ hi_minus_lst_z            r=-0.613
    PC1 ↔ lst_c                     r=+0.610
    PC2 ↔ hi_c                      r=+0.575
    PC1 ↔ hi_c                      r=-0.365
    PC3 ↔ hi_minus_lst_z            r=-0.222

CHICAGO  (n=872)
  Explained variance: PC1=0.308  PC2=0.279  PC3=0.102  PC4=0.093  PC5=0.046  PC6=0.033  PC7=0.021
  Cumulative (PC1-7): 0.881
  Top PC-observable correlations:
    PC1 ↔ hi_minu

In [25]:

"""
SHAP on PC scores (per city)
- Load PC1-PC7 scores + gap_z
- Train RF per city on PCs
- SHAP TreeExplainer → which PCs drive the gap
- Save: outputs/tables/<city>_pc_shap_values.csv
        outputs/figures/pca/<city>_pc_shap_summary.png
"""
import os
import pandas as pd, numpy as np, shap, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor

ROOT     = Path(".").resolve()
print(ROOT)
OUT      = ROOT / "outputs"
FIG_PCA  = OUT / "figures" / "pca"
FIG_PCA.mkdir(parents=True, exist_ok=True)

mt = pd.read_csv("data/processed/modeling_table.csv", dtype={"GEOID": str})
mt["GEOID"] = mt["GEOID"].str.zfill(11)
CITIES = sorted(mt["city"].unique().tolist())

def compute_gap_z(grp):
    def zs(s): return (s - s.mean()) / s.std() if s.std() > 0 else s * 0
    return zs(grp["hi_c"]) - zs(grp["lst_c"])

mt["gap_z"] = mt.groupby("city", group_keys=False).apply(
    compute_gap_z, include_groups=False
)

all_pc_shap = []

for city in CITIES:
    pc_path = OUT /"tables" / f"{city}_pc_scores.csv"
    # print(pc_path)
    if not pc_path.exists():
        print(f"{city}: no PC scores, skip"); continue

    pc_df = pd.read_csv(pc_path, dtype={"GEOID": str})
    pc_df["GEOID"] = pc_df["GEOID"].str.zfill(11)

    PC_COLS = [c for c in pc_df.columns if c.startswith("PC")]

    city_mt = mt[mt["city"] == city][["GEOID", "gap_z"]].copy()
    merged  = pc_df.merge(city_mt, on="GEOID", how="inner").dropna(subset=PC_COLS + ["gap_z"])

    X = merged[PC_COLS]
    y = merged["gap_z"]

    if len(merged) < 20:
        print(f"{city}: too few rows, skip"); continue

    rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
    rf.fit(X, y)
    print(f"\n{city.upper()}  train R²={rf.score(X, y):.3f}  n={len(merged)}")

    explainer   = shap.TreeExplainer(rf)
    shap_values = explainer.shap_values(X)

    # Save SHAP values
    shap_df = pd.DataFrame(shap_values, columns=PC_COLS)
    shap_df.insert(0, "GEOID", merged["GEOID"].values)
    shap_df.to_csv(OUT / "tables" / f"{city}_pc_shap_values.csv", index=False)

    # Summary plot
    plt.figure(figsize=(7, 5))
    shap.summary_plot(shap_values, X, show=False)
    plt.title(f"{city.replace('_', ' ').title()} — SHAP on PC scores", fontsize=12)
    plt.tight_layout()
    plt.savefig(FIG_PCA / f"{city}_pc_shap_summary.png", dpi=200)
    plt.close()

    # Print top PCs
    importance = pd.Series(
        np.mean(np.abs(shap_values), axis=0), index=PC_COLS
    ).sort_values(ascending=False)
    print("  PC importance:", "  ".join(f"{k}={v:.3f}" for k, v in importance.items()))

    for pc, val in importance.items():
        all_pc_shap.append({"city": city, "pc": pc, "mean_abs_shap": round(val, 4)})

pd.DataFrame(all_pc_shap).to_csv(OUT / "tables" / "all_cities_pc_shap_importance.csv", index=False)
print("\nSaved:")
print("  outputs/tables/<city>_pc_shap_values.csv")
print("  outputs/figures/pca/<city>_pc_shap_summary.png")
print("  outputs/tables/all_cities_pc_shap_importance.csv")


/Users/chenchenmengmeng/Documents/Development/Projects/Sigspatial/heat-exposure-hotspot-mismatch

ATLANTA  train R²=0.962  n=223
  PC importance: PC1=0.764  PC2=0.407  PC4=0.315  PC3=0.198  PC6=0.141  PC7=0.066  PC5=0.060

BOSTON  train R²=0.919  n=191
  PC importance: PC1=0.643  PC5=0.244  PC2=0.199  PC6=0.116  PC3=0.104  PC4=0.096  PC7=0.067

CHICAGO  train R²=0.919  n=872
  PC importance: PC1=0.263  PC2=0.165  PC4=0.125  PC6=0.096  PC7=0.074  PC5=0.053  PC3=0.047

DALLAS  train R²=0.926  n=449
  PC importance: PC1=0.625  PC2=0.333  PC4=0.213  PC3=0.108  PC7=0.065  PC5=0.061  PC6=0.056

HOUSTON  train R²=0.925  n=549
  PC importance: PC1=0.376  PC2=0.282  PC3=0.169  PC7=0.121  PC4=0.082  PC6=0.058  PC5=0.040

LAS_VEGAS  train R²=0.883  n=458
  PC importance: PC3=0.193  PC6=0.149  PC7=0.097  PC4=0.068  PC2=0.060  PC5=0.040  PC1=0.030

LOS_ANGELES  train R²=0.947  n=1080
  PC importance: PC4=0.303  PC5=0.239  PC1=0.192  PC7=0.186  PC3=0.177  PC2=0.136  PC6=0.124

MIAMI  train R²=0.944 